In [27]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import folium
from folium.plugins import HeatMap
import plotly.express as px
from sklearn.neighbors import BallTree
import branca.colormap as cm


In [28]:
df = pd.read_parquet(
    r"../data/processed/v2/housing_residential_processed.parquet"
).copy()

In [29]:
df.head()

,address,longitude,latitude,area,room_count,floor,floor_count,market_type,flat_type,ceiling_height,build_year,balcony,price,price_per_square_meter,date
0,"Коммунистическая, д. 26",37.580768,55.505888,44.7,2,2,5,secondary,flat,NaN,<NA>,False,3500000.0,78299.8,2025-06-09
1,"Рябиновая, д. 3, к. 1",37.424902,55.718709,19.5,0,1,15,secondary,studio,NaN,<NA>,False,6100000.0,312820.5,2025-05-21
2,"Большая Очаковская, д. 3",37.465910,55.688563,61.4,2,18,24,secondary,flat,NaN,<NA>,False,26000000.0,423452.8,2025-05-06
3,"квартал Введенское, д. 1Б",37.617644,55.755819,43.0,2,8,9,secondary,flat,NaN,<NA>,False,6711000.0,156069.8,2025-05-21
4,д. 14,37.617644,55.755819,44.0,2,2,5,secondary,flat,NaN,<NA>,False,3500000.0,79545.5,2025-06-10


In [30]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1336403 entries, 0 to 1336402
Data columns (total 15 columns):
 #   Column                  Non-Null Count    Dtype         
---  ------                  --------------    -----         
 0   address                 1336403 non-null  string        
 1   longitude               1336403 non-null  float64       
 2   latitude                1336403 non-null  float64       
 3   area                    1336403 non-null  float64       
 4   room_count              1190602 non-null  Int64         
 5   floor                   1336403 non-null  Int64         
 6   floor_count             126801 non-null   Int64         
 7   market_type             1336403 non-null  string        
 8   flat_type               1336403 non-null  string        
 9   ceiling_height          34998 non-null    float64       
 10  build_year              1232483 non-null  Int64         
 11  balcony                 1336403 non-null  bool          
 12  price                   1

In [31]:
df_primary = df[df["market_type"] == "primary"]
df_secondary = df[df["market_type"] == "secondary"]

Сделаю хитмап по точкам. Слишком много точек, поэтому хитмап, который показывает плотность.

In [ ]:
m = folium.Map(location=[55.75, 37.62], zoom_start=10, tiles=None)

folium.TileLayer("CartoDB positron", opacity=0.4).add_to(m)

# --- Use ONE gradient everywhere (HeatMap + legend) ---
gradient = {
    0.0: "#0000ff",  # blue
    0.25: "#00ffff", # cyan
    0.5: "#00ff00",  # lime
    0.75: "#ffff00", # yellow
    1.0: "#ff0000",  # red
}

# --- Make color meaning consistent across layers ---
# HeatMap intensity is based on kernel density; to align colors, force same max_val.
max_val = max(len(df_primary), len(df_secondary))  # simple, stable shared scale

fg_primary = folium.FeatureGroup(name="Primary", show=True)
HeatMap(
    data=df_primary[["latitude", "longitude"]].values.tolist(),
    radius=6,
    blur=6,
    max_zoom=13,
    min_opacity=0.4,
    gradient=gradient,
).add_to(fg_primary)
fg_primary.add_to(m)

fg_secondary = folium.FeatureGroup(name="Secondary", show=False)
HeatMap(
    data=df_secondary[["latitude", "longitude"]].values.tolist(),
    radius=6,
    blur=6,
    max_zoom=13,
    min_opacity=0.4,
    gradient=gradient,
).add_to(fg_secondary)
fg_secondary.add_to(m)

# --- Legend that actually matches the heatmap colors ---
legend = cm.LinearColormap(
    colors=[gradient[0.0], gradient[0.25], gradient[0.5], gradient[0.75], gradient[1.0]],
    vmin=0,
    vmax=max_val,
    caption="Heat intensity (relative; shared scale)",
)
legend.add_to(m)

folium.LayerControl(collapsed=False).add_to(m)
m.save("../resources/primary_vs_secondary_geo_heatmaps.html")

Создам новый признак - средневзвешенная по расстоянию цена 30 ближайших к точке точек.

При подсчете не учитывается сама точка.

Вес $w=e^{-\frac{d}{h}}$, где $d$ - расстрояние до соседа, h - параметр.

In [33]:
def mean_knn_by_geo(df, k: int, h: float, coords, tree, column_name, new_column_name: str):
    k += 1  # берем на одну больше, так как саму точку не учитываем

    # 2. строим дерево

    dist, ind = tree.query(coords, k=k)
    dist_m = dist * 6371000

    prices = df[column_name].values

    # удаляем первый столбец (сама точка)
    dist_m = dist_m[:, 1:]
    ind = ind[:, 1:]

    dist_m[dist_m == 0] = 1

    weights = np.exp(-dist_m / h)

    weighted_price = np.sum(weights * prices[ind], axis=1) / np.sum(weights, axis=1)

    df[new_column_name] = weighted_price

    return df

TODO: Подобрать $h$ через MAE. Для подбора использую 10% данных из-за ресурных ограничений.

Смотрю среднюю цену по ближайшим для общей цены и цены за квадратный метр на 10% данных.

In [34]:
n = 100_000
df_primary_sample = df_primary.sample(n=n, random_state=42)
df_secondary_sample = df_secondary.sample(n=n, random_state=42)

In [37]:
coords_primary_sample = np.radians(df_primary_sample[["latitude", "longitude"]].to_numpy(dtype=np.float32))
coords_secondary_sample = np.radians(df_secondary_sample[["latitude", "longitude"]].to_numpy(dtype=np.float32))
tree_primary_sample = BallTree(coords_primary_sample, metric="haversine", leaf_size=40)
tree_secondary_sample = BallTree(coords_secondary_sample, metric="haversine", leaf_size=40)

In [38]:
df_primary_sample = mean_knn_by_geo(df_primary_sample, 30, 500, coords_primary_sample, tree_primary_sample, "price", "knn_weighted_price")

In [39]:
df_secondary_sample = mean_knn_by_geo(df_secondary_sample, 30, 500, coords_secondary_sample, tree_secondary_sample, "price", "knn_weighted_price")

In [40]:
df_primary_sample = mean_knn_by_geo(df_primary_sample, 30, 500, coords_primary_sample, tree_primary_sample, "price_per_square_meter", "knn_weighted_price_per_square_meter")

In [41]:
df_secondary_sample = mean_knn_by_geo(df_secondary_sample, 30, 500, coords_secondary_sample, tree_secondary_sample, "price_per_square_meter", "knn_weighted_price_per_square_meter")

Пока не хватает ресурсов для всех точек.

```
coords_primary = np.radians(df_primary[["latitude", "longitude"]].to_numpy(dtype=np.float32))
coords_secondary = np.radians(df_secondary[["latitude", "longitude"]].to_numpy(dtype=np.float32))
tree_primary = BallTree(coords_primary, metric="haversine", leaf_size=40)
tree_secondary = BallTree(coords_secondary, metric="haversine", leaf_size=40)

df_primary = mean_knn_by_geo(df_primary, 30, 500, coords_primary, tree_primary, "price", "knn_weighted_price")
```

In [110]:
import numpy as np
import folium
from folium.plugins import HeatMap
from branca.colormap import linear

def make_heatmap_by_price(df_primary, df_secondary, column_name: str, save_path: str):
    m = folium.Map(location=[55.75, 37.62], zoom_start=10, tiles=None)
    folium.TileLayer("CartoDB positron", opacity=0.4).add_to(m)

    # значения в млн
    primary_vals = df_primary[column_name].dropna().to_numpy() / 1_000_000
    secondary_vals = df_secondary[column_name].dropna().to_numpy() / 1_000_000
    vals = np.concatenate([primary_vals, secondary_vals])

    vmin, vmax = float(vals.min()), float(vals.max())

    # Легенда: 6 интервалов
    colormap = linear.YlOrRd_09.scale(vmin, vmax).to_step(6)
    colormap.caption = f"{column_name} (M ₽)"
    colormap.add_to(m)

    # Градиент для HeatMap из тех же 6 цветов (чтобы совпадало визуально)
    # Индексы 0..1
    n_steps = 6
    gradient = {}
    for i in range(n_steps):
        x = i / (n_steps - 1)  # 0..1
        # берем цвет ровно по шкале colormap (она уже stepped)
        gradient[x] = colormap(vmin + x * (vmax - vmin))

    def build_heat(df):
        return (
            df[["latitude", "longitude", column_name]]
            .dropna()
            .assign(weight=lambda x: x[column_name] / 1_000_000)
            [["latitude", "longitude", "weight"]]
            .to_numpy()
            .tolist()
        )

    heat_primary = build_heat(df_primary)
    heat_secondary = build_heat(df_secondary)

    fg1 = folium.FeatureGroup(name="Primary", show=True)
    fg2 = folium.FeatureGroup(name="Secondary", show=False)

    # ВАЖНО: max_val фиксирует нормировку (иначе auto по слою)
    HeatMap(
        heat_primary,
        radius=15,
        min_opacity=0.2,
        gradient=gradient,
    ).add_to(fg1)

    HeatMap(
        heat_secondary,
        radius=15,
        min_opacity=0.2,
        gradient=gradient,
    ).add_to(fg2)

    fg1.add_to(m)
    fg2.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)

    m.save(save_path)

In [111]:
make_heatmap_by_price(df_primary_sample, df_secondary_sample, column_name="knn_weighted_price", save_path="../resources/primary_vs_secondary_knn_price_heatmaps.html")

In [112]:
make_heatmap_by_price(df_primary_sample, df_secondary_sample, column_name="knn_weighted_price_per_square_meter", save_path="../resources/primary_vs_secondary_knn_price_per_square_meter_heatmaps.html")

Теперь посмотрим отношение knn цен первички к вторичке.

In [106]:
def compute_ratio_knn(
    df_primary,
    df_secondary,
    coords_primary,
    tree_secondary,
    column_name,
    new_column_name,
    k,
):
    dist, ind = tree_secondary.query(coords_primary, k=k)

    secondary_vals = df_secondary[column_name].values[ind]
    secondary_mean = secondary_vals.mean(axis=1)

    primary_vals = df_primary[column_name].values

    ratio = primary_vals / secondary_mean

    df_primary[new_column_name] = ratio
    
    return df_primary

In [65]:
df_primary_sample = compute_ratio_knn(
    df_primary=df_primary_sample,
    df_secondary=df_secondary_sample,
    coords_primary=coords_primary_sample,
    tree_secondary=tree_secondary_sample,
    column_name="knn_weighted_price",
    new_column_name="primary_to_secondary_price_ratio",
    k=30
)

In [66]:
df_primary_sample = compute_ratio_knn(
    df_primary=df_primary_sample,
    df_secondary=df_secondary_sample,
    coords_primary=coords_primary_sample,
    tree_secondary=tree_secondary_sample,
    column_name="knn_weighted_price_per_square_meter",
    new_column_name="primary_to_secondary_m2_ratio",
    k=30
)

In [85]:
def make_heatmap_ratio(
    df_primary,
    ratio_column: str,
    save_path: str,
    radius=15,
    clip_quantiles=(0.02, 0.98),
):
    m = folium.Map(location=[55.75, 37.62], zoom_start=10, tiles=None)

    folium.TileLayer(
    "CartoDB positron",
    opacity=0.4
    ).add_to(m)

    s = df_primary[["latitude", "longitude", ratio_column]].dropna().copy()
    s = s[np.isfinite(s[ratio_column])]
    s = s[s[ratio_column] > 0]

    # Квантили для обрезки выбросов
    q_low = float(s[ratio_column].quantile(clip_quantiles[0]))
    q_high = float(s[ratio_column].quantile(clip_quantiles[1]))

    # Симметрия вокруг 1
    max_dev = max(abs(1 - q_low), abs(q_high - 1))
    vmin = 1 - max_dev
    vmax = 1 + max_dev

    # Нормализация значений в [0, 1] для HeatMap
    s["w"] = (s[ratio_column] - vmin) / (vmax - vmin)
    s["w"] = s["w"].clip(0, 1)

    heat_data = s[["latitude", "longitude", "w"]].values.tolist()

    # Градиент HeatMap: синий -> белый -> красный
    # ключи — позиции 0..1
    gradient = {
        0.0: "#2c7bb6",   # синий
        0.5: "#ffffbf",   # почти белый/кремовый (лучше виден на карте)
        1.0: "#d7191c",   # красный
    }

    HeatMap(
        heat_data,
        radius=radius,
        gradient=gradient,
        min_opacity=0.2,
        blur=15,
        max_zoom=12,
    ).add_to(m)

    # Легенда (соответствует vmin..vmax и тому же смыслу)
    colormap = cm.LinearColormap(
        colors=[gradient[0.0], gradient[0.5], gradient[1.0]],
        vmin=vmin,
        vmax=vmax,
    )
    colormap.caption = f"{ratio_column} (Primary/Secondary), 1 = parity"
    colormap.add_to(m)

    m.save(save_path)

In [86]:
make_heatmap_ratio(
    df_primary_sample,
    ratio_column="primary_to_secondary_price_ratio",
    save_path="../resources/ratio_price_heatmap.html"
)